# Exp1 vs Exp3 — Neuron Comparison
Direct visual comparison of reward-sensitive neurons (Exp1) vs anhedonia-changed neurons (Exp3).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size']  = 11

N_LAYERS  = 28
N_NEURONS = 18944

In [ ]:
# ── Load Exp1 CSVs ────────────────────────────────────────────────────────────
df_core   = pd.read_csv('data/master_incentive_core.csv')
df_money  = pd.read_csv('data/universal_money_neurons.csv')
df_reward = pd.read_csv('data/universal_reward_neurons.csv')

set_core   = set(zip(df_core['layer'],   df_core['neuron']))
set_money  = set(zip(df_money['layer'],  df_money['neuron']))
set_reward = set(zip(df_reward['layer'], df_reward['neuron']))

# ── Load Exp3 activations ─────────────────────────────────────────────────────
neu_normal    = np.load('activations/activations_neurons_normal.npy')    # (100,28,18944)
neu_anhedonic = np.load('activations/activations_neurons_anhedonic.npy')

diff_signed = neu_anhedonic.mean(axis=0) - neu_normal.mean(axis=0)      # (28,18944)
diff_abs    = np.abs(diff_signed)

pooled_std  = np.sqrt((neu_normal.std(axis=0)**2 + neu_anhedonic.std(axis=0)**2) / 2 + 1e-8)
effect_size = diff_signed / pooled_std

# Exp3 top-500 changed neurons
flat_idx   = np.argsort(diff_abs.flatten())[::-1][:500]
exp3_layers = flat_idx // N_NEURONS
exp3_neurs  = flat_idx %  N_NEURONS
exp3_deltas = diff_signed[exp3_layers, exp3_neurs]
set_exp3    = set(zip(exp3_layers, exp3_neurs))
set_exp3_up   = {(l,n) for l,n in set_exp3 if diff_signed[l,n] > 0}
set_exp3_down = {(l,n) for l,n in set_exp3 if diff_signed[l,n] < 0}

print('Loaded.')
print(f'Exp1 core: {len(set_core)} | money: {len(set_money)} | reward: {len(set_reward)}')
print(f'Exp3 top-500: {len(set_exp3_up)} UP, {len(set_exp3_down)} DOWN')

## Plot 1 — Layer Distribution: Exp1 vs Exp3 Side by Side

In [ ]:
# Count neurons per layer for each set
def count_per_layer(neuron_set, n_layers=28):
    counts = np.zeros(n_layers, dtype=int)
    for l, n in neuron_set:
        counts[l] += 1
    return counts

layers = np.arange(N_LAYERS)

core_per_layer   = count_per_layer(set_core)
money_per_layer  = count_per_layer(set_money)
reward_per_layer = count_per_layer(set_reward)
exp3_up_per_layer   = count_per_layer(set_exp3_up)
exp3_down_per_layer = count_per_layer(set_exp3_down)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ── TOP: Exp1 neuron counts per layer ──
ax = axes[0]
w = 0.28
ax.bar(layers - w,   core_per_layer,   w, label=f'Exp1 Incentive Core ({len(set_core)})',   color='#1565C0', alpha=0.85)
ax.bar(layers,       money_per_layer,  w, label=f'Exp1 Money ({len(set_money)})',           color='#FF8F00', alpha=0.85)
ax.bar(layers + w,   reward_per_layer, w, label=f'Exp1 Reward ({len(set_reward)})',         color='#6A1B9A', alpha=0.85)
ax.set_ylabel('# Neurons')
ax.set_title('Exp1 — Incentive-Sensitive Neurons per Layer', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(layers)

# ── BOTTOM: Exp3 anhedonia-changed neurons per layer ──
ax = axes[1]
ax.bar(layers,  exp3_up_per_layer,    0.4, label='Exp3 UP under anhedonia',   color='#D32F2F', alpha=0.85)
ax.bar(layers, -exp3_down_per_layer,  0.4, label='Exp3 DOWN under anhedonia', color='#1976D2', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('# Neurons (↑ above / ↓ below)')
ax.set_xlabel('Layer')
ax.set_title('Exp3 — Most Changed Neurons (Top 500) per Layer under Anhedonic Prompt', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(layers)

plt.suptitle('Layer Distribution: Exp1 Reward Neurons vs Exp3 Anhedonia-Changed Neurons',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_layer_distribution_comparison.png', bbox_inches='tight')
plt.show()

## Plot 2 — Scatter: Every Neuron as a Point (Layer vs Neuron Index)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ── LEFT: Exp1 neurons ──
ax = axes[0]
# Plot each Exp1 set as a different color/marker
# Core = blue dots, money-only = orange, reward-only = purple, money+reward = green
money_only  = set_money  - set_core - set_reward
reward_only = set_reward - set_core - set_money
money_and_reward = (set_money & set_reward) - set_core

def scatter_set(ax, neuron_set, color, label, marker='.', size=4, alpha=0.5, zorder=1):
    if not neuron_set: return
    ls, ns = zip(*neuron_set)
    ax.scatter(ns, ls, c=color, s=size, alpha=alpha, marker=marker,
               label=label, zorder=zorder, linewidths=0)

scatter_set(ax, money_only,       '#FF8F00', f'Money only ({len(money_only)})',           size=3,  alpha=0.4)
scatter_set(ax, reward_only,      '#6A1B9A', f'Reward only ({len(reward_only)})',         size=3,  alpha=0.4)
scatter_set(ax, money_and_reward, '#2E7D32', f'Money+Reward ({len(money_and_reward)})',   size=5,  alpha=0.6)
scatter_set(ax, set_core,         '#D32F2F', f'Incentive Core ({len(set_core)})',         size=8,  alpha=0.7, zorder=3)

ax.set_xlabel('Neuron Index', fontsize=11)
ax.set_ylabel('Layer', fontsize=11)
ax.set_yticks(range(N_LAYERS))
ax.set_title('Exp1 — Incentive-Sensitive Neurons\n(Layer × Neuron Index space)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, markerscale=3)
ax.set_xlim(-100, N_NEURONS + 100)
ax.set_ylim(-1, N_LAYERS)
ax.grid(alpha=0.15)

# ── RIGHT: Exp3 neurons ──
ax = axes[1]
# Background: all top-500 Exp3 neurons
exp3_up_l,   exp3_up_n   = zip(*set_exp3_up)   if set_exp3_up   else ([], [])
exp3_down_l, exp3_down_n = zip(*set_exp3_down) if set_exp3_down else ([], [])
ax.scatter(exp3_up_n,   exp3_up_l,   c='#FFCDD2', s=8,  alpha=0.6, label=f'Exp3 UP ({len(set_exp3_up)})',   linewidths=0)
ax.scatter(exp3_down_n, exp3_down_l, c='#BBDEFB', s=8,  alpha=0.6, label=f'Exp3 DOWN ({len(set_exp3_down)})', linewidths=0)

# Overlay: neurons in BOTH Exp3 AND Exp1 core
overlap_up_core   = set_exp3_up   & set_core
overlap_down_core = set_exp3_down & set_core
overlap_up_any    = set_exp3_up   & (set_money | set_reward)
overlap_down_any  = set_exp3_down & (set_money | set_reward)

scatter_set(ax, overlap_up_any,    '#FF8F00', f'Also in Exp1 money/reward ↑ ({len(overlap_up_any)})',   size=30, alpha=0.8, zorder=3)
scatter_set(ax, overlap_down_any,  '#1565C0', f'Also in Exp1 money/reward ↓ ({len(overlap_down_any)})', size=30, alpha=0.8, zorder=3)
scatter_set(ax, overlap_up_core,   '#B71C1C', f'Also in Exp1 CORE ↑ ({len(overlap_up_core)})',          size=60, alpha=1.0, zorder=4, marker='*')
scatter_set(ax, overlap_down_core, '#0D47A1', f'Also in Exp1 CORE ↓ ({len(overlap_down_core)})',        size=60, alpha=1.0, zorder=4, marker='*')

ax.set_xlabel('Neuron Index', fontsize=11)
ax.set_ylabel('Layer', fontsize=11)
ax.set_yticks(range(N_LAYERS))
ax.set_title('Exp3 — Top 500 Anhedonia-Changed Neurons\nHighlighted: overlap with Exp1', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, markerscale=1.5)
ax.set_xlim(-100, N_NEURONS + 100)
ax.set_ylim(-1, N_LAYERS)
ax.grid(alpha=0.15)

plt.suptitle('Exp1 vs Exp3 — Neuron Space (Layer × Index)\nStars = neurons appearing in BOTH experiments',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_scatter_layer_neuron.png', bbox_inches='tight')
plt.show()

## Plot 3 — Exact Shared Neurons: Zoomed Layer View

In [ ]:
# For each layer that has shared neurons: show Exp1 core density vs Exp3 change magnitude
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ── TOP: Exp1 core neuron density across neuron index, per layer (stacked) ──
ax = axes[0]
# Show as horizontal strip per layer: each neuron in core gets a tick mark
for l in range(N_LAYERS):
    core_in_layer = [n for (ll, n) in set_core if ll == l]
    if core_in_layer:
        ax.scatter(core_in_layer, [l]*len(core_in_layer),
                   c='#D32F2F', s=4, alpha=0.4, linewidths=0)

ax.set_ylabel('Layer')
ax.set_yticks(range(N_LAYERS))
ax.set_title('Exp1 — Incentive Core Neuron Positions (each dot = one neuron)', fontweight='bold')
ax.set_xlim(0, N_NEURONS)
ax.grid(alpha=0.15)

# ── BOTTOM: Exp3 delta heatmap across neuron index per layer ──
ax = axes[1]
for l in range(N_LAYERS):
    exp3_in_layer_down = [n for (ll, n) in set_exp3_down if ll == l]
    exp3_in_layer_up   = [n for (ll, n) in set_exp3_up   if ll == l]
    if exp3_in_layer_down:
        ax.scatter(exp3_in_layer_down, [l]*len(exp3_in_layer_down),
                   c='#1565C0', s=6, alpha=0.6, linewidths=0)
    if exp3_in_layer_up:
        ax.scatter(exp3_in_layer_up, [l]*len(exp3_in_layer_up),
                   c='#D32F2F', s=6, alpha=0.6, linewidths=0)

    # Star the shared ones
    shared_down = [n for (ll,n) in (set_exp3_down & set_core) if ll == l]
    shared_up   = [n for (ll,n) in (set_exp3_up   & set_core) if ll == l]
    if shared_down:
        ax.scatter(shared_down, [l]*len(shared_down),
                   c='#0D47A1', s=120, marker='*', zorder=5, label='Shared ↓' if l==min(ll for ll,_ in set_exp3_down & set_core) else '')
    if shared_up:
        ax.scatter(shared_up, [l]*len(shared_up),
                   c='#B71C1C', s=120, marker='*', zorder=5)

ax.set_xlabel('Neuron Index')
ax.set_ylabel('Layer')
ax.set_yticks(range(N_LAYERS))
ax.set_title('Exp3 — Top 500 Changed Neurons  (red=UP, blue=DOWN, stars=shared with Exp1 Core)', fontweight='bold')
ax.set_xlim(0, N_NEURONS)
ax.grid(alpha=0.15)

plt.suptitle('Direct Comparison: Where Are the Neurons in the Layer × Index Space?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_direct_comparison.png', bbox_inches='tight')
plt.show()

## Table — Exact Shared Neurons with Full Stats

In [ ]:
rows = []
for rank, (l, n) in enumerate(zip(exp3_layers, exp3_neurs), 1):
    in_c = (l,n) in set_core
    in_m = (l,n) in set_money
    in_r = (l,n) in set_reward
    if in_c or in_m or in_r:
        rows.append({
            'exp3_rank':   rank,
            'layer':       int(l),
            'neuron':      int(n),
            'delta':       round(float(diff_signed[l,n]), 4),
            'effect_size': round(float(effect_size[l,n]),  4),
            'direction':   'UP' if diff_signed[l,n] > 0 else 'DOWN',
            'in_core':     in_c,
            'in_money':    in_m,
            'in_reward':   in_r,
        })

df_shared = pd.DataFrame(rows).sort_values('exp3_rank')

print(f'Total shared neurons (Exp3 top-500 ∩ any Exp1 set): {len(df_shared)}')
print(f'  In core   : {df_shared["in_core"].sum()}')
print(f'  DOWN      : {(df_shared["direction"]=="DOWN").sum()}')
print(f'  UP        : {(df_shared["direction"]=="UP").sum()}')
print()
print(df_shared.to_string(index=False))

df_shared.to_csv('shared_neurons_exp1_exp3.csv', index=False)
print('\nSaved → shared_neurons_exp1_exp3.csv')